# Gathering và hierarchy theo version

Tìm root, duyệt include theo thứ tự và dựng cây occurrence kèm source map. Các leaf gồm sentence, block math và figure/table; list giữ cấu trúc item. Bibliography được xử lý riêng; acknowledgements và appendix được giữ.

Ví dụ lấy từ tối đa hai paper đầu; coverage tính trên toàn manifest.


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from crawler.reporting import load_m2, run_table
m2 = load_m2()
manifest, stats = m2['manifest'], m2['stats']
print('Input:', m2['directory'])
print('Phạm vi:', stats.get('publications'), 'paper;', stats.get('versions'), 'version')
print('Run:', stats.get('run_id'), '; status:', stats.get('status'))


In [ ]:
coverage=m2.get('project_validation')
if coverage:
    print('Kiểm tra cấu trúc 5 bảng:',coverage['structural_valid'])
    print('Đã xử lý đầy đủ phạm vi M1:',coverage['processing_complete'])
    display(pd.DataFrame([coverage['coverage']]).drop(columns=['missing_sample'],errors='ignore'))
else:
    print('Chưa có validation 5 bảng cho run này; coverage bên dưới chỉ thuộc input parser hiện tại.')
display(manifest.reindex(columns=['arxiv_id','version','status','has_tex','root','root_selection','elements_before','bibtex_before','counts_measured']))
display(pd.DataFrame(stats.get('publication_coverage', [])))
display(pd.DataFrame(stats.get('diagnostics', [])))
manifest['status'].value_counts().plot.bar(title='Trạng thái parse theo version — phạm vi input đã nêu',ylabel='Số version')
plt.tight_layout(); plt.show()
if m2['working']:
    example=m2['working'][0]
    print('Ví dụ paper:',example['arxiv_id'])
    occurrences=pd.DataFrame(example.get('occurrences',[]))
    display(occurrences.reindex(columns=['version','node_id','parent_id','kind','content']).head(15))
    for version, gathered in list(example.get('gathering',{}).items())[:1]:
        print('Compilation root, version',version,':',gathered.get('root'))
        print('Các file thuộc compilation path:',gathered.get('files'))
        display(pd.DataFrame(gathered.get('source_map',[])[:10]))


## Giới hạn

Diagnostics ghi source thiếu, root mơ hồ, include lỗi và cú pháp chưa hỗ trợ. Cần đối chiếu thủ công các bố cục đặc biệt. Source thiếu và PDF-only là hai trạng thái riêng.
